# 14장. ONNX 파일 구성 들여다보기

In [1]:
# [준비 셀] 모든 노트북의 첫 코드 셀입니다. 항상 먼저 실행하세요.
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[1]          # docs/manual → 프로젝트 루트
assert (ROOT / "src/bearing_kd").exists(), "docs/manual 폴더에서 노트북을 열어야 합니다"
CODE = ROOT / "docs/manual/code"                 # 챕터별 완성 스크립트 위치
sys.path.insert(0, str(CODE))                    # 스크립트의 함수를 import 해서 쓰기 위해
print("프로젝트 루트 확인 완료:", ROOT.name)

프로젝트 루트 확인 완료: vib-onnx


13장에서 만든 `student.onnx` 안에는 무엇이 들어 있을까요? 이 장에서는 ONNX 파일을 한 겹씩 벗겨 봅니다.

## 14.1 한 장으로 보는 ONNX 파일 구조

ONNX 파일은 **protobuf**(구글이 만든 이진 데이터 형식)로 저장된 트리 구조입니다.

```text
ModelProto                          ← 파일 전체
├─ ir_version = 10                  ← 파일 형식 버전
├─ opset_import = [ai.onnx v18]     ← 사용하는 연산 목록의 버전
├─ producer = pytorch 2.14.0        ← 만든 도구
└─ graph: GraphProto                ← 계산 그래프 (요리법)
   ├─ input       [x_enc, input_mask]     이름, 모양, 자료형
   ├─ output      [logits]
   ├─ initializer [가중치 109개]           이름, 모양, 값 (또는 외부 파일 위치)
   ├─ node        [연산 435개]             연산 종류, 입력 이름들 → 출력 이름들
   └─ value_info  [중간 결과의 모양]
```

핵심은 **노드(node)**와 **이니셜라이저(initializer)**입니다.
- **노드**: `MatMul`, `Add`, `Softmax` 같은 연산 하나. "어떤 이름의 값을 받아서, 어떤 이름의 값을 만든다"
- **이니셜라이저**: 학습된 가중치처럼 **미리 정해진 값**
- 노드들은 **이름으로 연결**됩니다. 한 노드의 출력 이름이 다른 노드의 입력 이름이면 그 둘이 이어집니다.

## 14.2 손으로 만드는 가장 작은 ONNX

2장의 선형 분류기 `y = x·W + b`를 ONNX로 직접 만들어 봅니다.

```text
x [1,2] ──▶ MatMul ──(xw)──▶ Add ──▶ y [1,2]
             ▲ W [2,2]         ▲ b [2]
          (initializer)     (initializer)
```

In [2]:
import onnx, numpy as np
import onnxruntime as ort
import ch14_inspect_onnx as ch14

tiny_path = ROOT / "outputs/manual/onnx/tiny.onnx"
tiny = ch14.build_tiny(tiny_path)
print(onnx.printer.to_text(tiny))   # 사람이 읽을 수 있는 글자로 출력

<
   ir_version: 10,
   opset_import: ["" : 18],
   producer_name: "manual"
>
tiny_linear (float[1,2] x) => (float[1,2] y) 
   <float[2,2] W =  {1,-1,0.5,2}, float[2] b =  {0.1,-0.1}>
{
   [matmul] xw = MatMul (x, W)
   [add] y = Add (xw, b)
}


In [3]:
sess = ort.InferenceSession(str(tiny_path), providers=["CPUExecutionProvider"])
y = sess.run(None, {"x": np.array([[1.0, 2.0]], dtype=np.float32)})[0]
print("ONNX Runtime 결과:", y.round(4).tolist())
print("직접 계산      :", (np.array([[1.0, 2.0]]) @ np.array([[1.0, -1.0], [0.5, 2.0]]) + np.array([0.1, -0.1])).round(4).tolist())
print("\n파일 헤더: IR", tiny.ir_version, "/ opset", tiny.opset_import[0].version, "/ 파일 크기", tiny_path.stat().st_size, "바이트")

ONNX Runtime 결과: [[2.0999999046325684, 2.9000000953674316]]
직접 계산      : [[2.1, 2.9]]

파일 헤더: IR 10 / opset 18 / 파일 크기 172 바이트


> **IR 버전과 opset은 다릅니다.** IR은 "파일 형식"의 버전, opset은 "연산 목록"의 버전입니다. 이 매뉴얼을 만들며 실제로 겪은 문제: `onnx 1.23`이 기본으로 IR 14 파일을 만드는데 `onnxruntime 1.30`은 IR 13까지만 읽어서 `Unsupported model IR version: 14` 에러가 났습니다. 그래서 `ir_version=10`을 지정했습니다(PyTorch가 만드는 값과 같음).

## 14.3 실제 Student ONNX 열어 보기

`load_external_data=False`로 열면 86 MB 가중치는 읽지 않고 **구조만** 빠르게 볼 수 있습니다.

In [4]:
model = ch14.describe(ROOT / "outputs/manual/onnx/student.onnx")

■ 헤더: IR 버전 10, opset [('ai.onnx', 18)], 만든 도구 pytorch 2.14.0
■ 입력: [('x_enc', [1, 1, 512], 'float32'), ('input_mask', [1, 512], 'float32')]
■ 출력: [('logits', [1, 2], 'float32')]
■ 노드(연산) 435개, 종류 30가지:
    Mul 77, MatMul 73, Add 61, Reshape 37, Transpose 33, Pow 26, Sqrt 18, ReduceMean 17, Reciprocal 17, Cast 8, Softmax 8, Tanh 8, Unsqueeze 7, ReduceSum 6, Where 6, And 4, Div 4, Not 3, IsNaN 3, Sub 3, Expand 3, Greater 2, IsInf 2, Clip 2, Gather 2, Less 1, Equal 1, Tile 1, Slice 1, Gemm 1
■ 가중치(initializer) 109개, 숫자 21,507,123개 (86.0 MB)
   그중 외부 파일에 저장 81개, .onnx 안에 직접 저장 28개(작은 상수)
   외부 데이터 예: model.patch_embedding.mask_embedding → 파일 'student.onnx.data', 시작 0바이트, 길이 2048바이트
■ 가장 큰 가중치 5개:
   model.patch_embedding.position_embedding.pe      [1, 5000, 512]
   val_166                                          [512, 1024]
   val_171                                          [512, 1024]
   val_172                                          [1024, 512]
   val_208                            

### 연산(노드) 종류 읽기

| 연산 | 개수 | 어디에 쓰이나 |
|---|---|---|
| `MatMul` | 73 | 행렬 곱. attention의 q/k/v/o, feed-forward 층 (모델 계산의 대부분) |
| `Mul`, `Add` | 138 | 곱하기, 더하기. 정규화, 잔차 연결(residual), 위치 bias |
| `Pow`, `ReduceMean`, `Sqrt`, `Reciprocal` | 78 | **RMSNorm** (T5의 층 정규화): x / √(평균(x²)) |
| `Softmax` | 8 | attention 점수 → 확률. 층마다 1개씩 = 8층 |
| `Tanh` | 8 | GELU 활성화 함수(근사식)에 들어감. 층마다 1개 |
| `Reshape`, `Transpose` | 70 | attention head 6개로 나눴다 합치기 |
| `Where`, `IsNaN`, `IsInf`, `ReduceSum` | | 13장에서 바꾼 RevIN과 head의 "유효한 칸만 평균" |
| `Gather` | 2 | 상대 위치 bias 표에서 값 꺼내기 |
| `Gemm` | 1 | 마지막 head 선형층 (행렬 곱 + 더하기를 한 번에) |

## 14.4 모듈별로 묶어 보기

노드 메타데이터에는 **어느 PyTorch 모듈에서 나온 연산인지**가 기록되어 있습니다. 이걸로 7장의 5단계 구조가 ONNX 안에서 어떻게 보이는지 확인합니다.

In [5]:
ch14.by_module(model)

   ① normalizer (RevIN)               노드  23개  (ReduceSum 3, Div 3, Not 2, And 2, Cast 2, Where 2)
   기타 (모양 변환 등)                       노드  16개  (Reshape 4, Where 3, And 2, IsNaN 1, Less 1, IsInf 1)
   ②③ 패치 + 임베딩                        노드  21개  (Cast 3, Unsqueeze 3, Gather 2, Expand 2, Mul 2, Add 2)
   ④ encoder (공통: 위치 bias, 마지막 norm)  노드  11개  (Mul 3, Unsqueeze 1, Cast 1, Sub 1, Pow 1, ReduceMean 1)
   ④ encoder 층 0                      노드  46개  (Mul 9, MatMul 9, Add 8, Reshape 4, Transpose 4, Pow 3)
   ④ encoder 층 1                      노드  44개  (Mul 9, MatMul 9, Add 7, Reshape 4, Transpose 4, Pow 3)
   ④ encoder 층 2                      노드  44개  (Mul 9, MatMul 9, Add 7, Reshape 4, Transpose 4, Pow 3)
   ④ encoder 층 3                      노드  44개  (Mul 9, MatMul 9, Add 7, Reshape 4, Transpose 4, Pow 3)
   ④ encoder 층 4                      노드  44개  (Mul 9, MatMul 9, Add 7, Reshape 4, Transpose 4, Pow 3)
   ④ encoder 층 5                      노드  44개  (Mul 9, MatMul 9, Add 7, Reshap

인코더 층 8개가 **똑같은 44개 노드 묶음**으로 반복됩니다. 한 층은 대략 이렇게 생겼습니다.

```text
입력 ─▶ RMSNorm ─▶ q, k, v = MatMul ×3 ─▶ head 6개로 Reshape/Transpose
     ─▶ q·kᵀ MatMul ─▶ + 상대 위치 bias ─▶ Softmax ─▶ ·v MatMul ─▶ o MatMul ─▶ + 입력(잔차)
     ─▶ RMSNorm ─▶ Feed-forward: MatMul(512→1024) ×2, GELU(Tanh) ─▶ MatMul(1024→512) ─▶ + (잔차) ─▶ 다음 층
```

## 14.5 가중치는 어디에? 외부 데이터 파일

- 큰 가중치 81개는 `student.onnx.data`에 **이어 붙여** 저장되고, `.onnx` 안에는 "`student.onnx.data` 파일의 몇 번째 바이트부터 몇 바이트"라는 **위치 정보만** 있습니다.
- 그래서 `.data` 파일의 **이름을 바꾸면** 위치 정보가 가리키는 파일을 못 찾아 로딩에 실패합니다.
- 작은 상수 28개(모양 정보, 스칼라)는 `.onnx` 안에 직접 들어 있습니다.

In [6]:
from onnx import TensorProto
ext = [t for t in model.graph.initializer if t.data_location == TensorProto.EXTERNAL]
for t in ext[:4]:
    info = {e.key: e.value for e in t.external_data}
    print(f"{t.name:55s} → {info['location']} [{info['offset']:>9s} ~ +{info['length']}] 바이트")
total = sum(int({e.key: e.value for e in t.external_data}["length"]) for t in ext)
data_file = ROOT / "outputs/manual/onnx/student.onnx.data"
print(f"\n외부 가중치 합계 {total:,} 바이트, .data 파일 크기 {data_file.stat().st_size:,} 바이트")

model.patch_embedding.mask_embedding                    → student.onnx.data [        0 ~ +2048] 바이트
model.encoder.block.0.layer.0.layer_norm.weight         → student.onnx.data [     2048 ~ +2048] 바이트
model.encoder.block.0.layer.1.layer_norm.weight         → student.onnx.data [     4096 ~ +2048] 바이트
model.encoder.block.1.layer.0.layer_norm.weight         → student.onnx.data [     6144 ~ +2048] 바이트

외부 가중치 합계 86,030,336 바이트, .data 파일 크기 86,065,152 바이트


## 14.6 PyTorch 모델과 무엇이 다른가

| | PyTorch Small (7장) | ONNX |
|---|---|---|
| 숫자 개수 | 35,338,434 | 21,507,123 |
| 글자용 임베딩 (32,128×512) | 있음 (16.4M) | **없음**: 계산에 쓰이지 않아 녹화되지 않음 |
| 위치 인코딩 표 (5,000×512) | buffer라 파라미터 수에 안 잡힘 | **있음** (2.56M): 계산에 쓰이므로 상수로 저장 |
| 가중치 이름 | `encoder.block.0.layer.1.DenseReluDense.wi_0.weight` | 일부는 `val_166`처럼 바뀜 |

이름이 바뀌는 이유: 선형층 `y = x·Wᵀ`를 ONNX `MatMul`로 바꾸면서 exporter가 **Wᵀ를 미리 계산(상수 접기, constant folding)**해 새 상수로 저장하기 때문입니다. 값은 같고 이름만 사라집니다.

In [7]:
names = [t.name for t in model.graph.initializer]
print("원래 이름이 남은 가중치:", sum(not n.startswith("val_") for n in names), "개  예:", [n for n in names if not n.startswith("val_")][:3])
print("이름이 바뀐 가중치    :", sum(n.startswith("val_") for n in names), "개  예:", [n for n in names if n.startswith("val_")][:3])

원래 이름이 남은 가중치: 26 개  예: ['model.patch_embedding.mask_embedding', 'model.encoder.block.0.layer.0.layer_norm.weight', 'model.encoder.block.0.layer.1.layer_norm.weight']
이름이 바뀐 가중치    : 83 개  예: ['val_9', 'val_44', 'val_66']


## 14.7 ⚠ 공개 전 필수: 노드 메타데이터 속 컴퓨터 경로

PyTorch exporter는 디버깅을 돕기 위해 노드마다 **그 연산을 만든 파이썬 소스 위치(stack trace)**를 남깁니다. 여기에는 `/Users/사용자이름/...` 같은 **컴퓨터 경로와 사용자 이름**이 들어 있습니다. 이 프로젝트는 실제로 이 상태의 ONNX를 한 번 공개 저장소에 올렸습니다.

In [8]:
node = model.graph.node[10]
print("노드 하나의 메타데이터 종류:", [p.key for p in node.metadata_props])
print("경로가 든 항목 수:", dict(ch14.scan_private(model)))

노드 하나의 메타데이터 종류: ['namespace', 'pkg.torch.onnx.class_hierarchy', 'pkg.torch.onnx.fx_node', 'pkg.torch.onnx.name_scopes', 'pkg.torch.onnx.stack_trace']
경로가 든 항목 수: {'pkg.torch.onnx.stack_trace': 435}


계산에는 쓰이지 않는 정보이므로 지워도 결과가 같습니다. 지운 뒤 다시 확인합니다.

In [9]:
removed = ch14.clean(ROOT / "outputs/manual/onnx/student.onnx", ROOT / "outputs/manual/onnx_clean/student.onnx")
cleaned = onnx.load(ROOT / "outputs/manual/onnx_clean/student.onnx", load_external_data=False)
print(f"stack trace {removed}개 삭제, 남은 경로:", dict(ch14.scan_private(cleaned)) or "없음")

from ch15_pi_deploy import predict_window
from ch06_dataset import WindowDataset
import json
test = WindowDataset([ROOT / f for f in json.loads((ROOT / "outputs/manual/split_holdout.json").read_text())["test"][:3]])
s1 = ort.InferenceSession(str(ROOT / "outputs/manual/onnx/student.onnx"), providers=["CPUExecutionProvider"])
s2 = ort.InferenceSession(str(ROOT / "outputs/manual/onnx_clean/student.onnx"), providers=["CPUExecutionProvider"])
d = max(np.abs(predict_window(s1, w) - predict_window(s2, w)).max() for w in test.windows)
print("정리 전/후 결과 차이:", d)

stack trace 435개 삭제, 남은 경로: 없음


정리 전/후 결과 차이: 0.0


## 14.8 그래프를 눈으로 보기: Netron

[Netron](https://netron.app)은 ONNX 그래프를 그림으로 보여 주는 무료 도구입니다. 웹사이트에 `student.onnx`를 끌어다 놓으면 노드 연결을 클릭하며 따라갈 수 있습니다. 파일은 브라우저 안에서만 열리지만, 민감한 모델이라면 데스크톱 앱 버전을 쓰세요. `.onnx.data`가 같은 폴더에 없으면 가중치 값은 보이지 않고 구조만 보입니다.

## 14장 완성 코드: `code/ch14_inspect_onnx.py`

In [10]:
%run code/ch14_inspect_onnx.py

[1] 가장 작은 ONNX
   x=[1,2] → y=[2.1, 2.9]  (직접 계산: [1·1+2·0.5+0.1, 1·(-1)+2·2-0.1] = [2.1, 2.9])

[2] Student ONNX 구성
■ 헤더: IR 버전 10, opset [('ai.onnx', 18)], 만든 도구 pytorch 2.14.0
■ 입력: [('x_enc', [1, 1, 512], 'float32'), ('input_mask', [1, 512], 'float32')]
■ 출력: [('logits', [1, 2], 'float32')]
■ 노드(연산) 435개, 종류 30가지:
    Mul 77, MatMul 73, Add 61, Reshape 37, Transpose 33, Pow 26, Sqrt 18, ReduceMean 17, Reciprocal 17, Cast 8, Softmax 8, Tanh 8, Unsqueeze 7, ReduceSum 6, Where 6, And 4, Div 4, Not 3, IsNaN 3, Sub 3, Expand 3, Greater 2, IsInf 2, Clip 2, Gather 2, Less 1, Equal 1, Tile 1, Slice 1, Gemm 1
■ 가중치(initializer) 109개, 숫자 21,507,123개 (86.0 MB)
   그중 외부 파일에 저장 81개, .onnx 안에 직접 저장 28개(작은 상수)
   외부 데이터 예: model.patch_embedding.mask_embedding → 파일 'student.onnx.data', 시작 0바이트, 길이 2048바이트
■ 가장 큰 가중치 5개:
   model.patch_embedding.position_embedding.pe      [1, 5000, 512]
   val_166                                          [512, 1024]
   val_171                                       

In [11]:
from IPython.display import Code
Code(filename=str(CODE / "ch14_inspect_onnx.py"), language="python")

"""14장 완성 코드: ONNX 파일이 어떻게 구성되어 있는지 들여다보고, 공개 전에 개인 정보를 지운다.

    1) 손으로 아주 작은 ONNX(y = x·W + b)를 만들어 구조 익히기
    2) 실제 Student ONNX: 헤더, 입출력, 노드(연산), 가중치(initializer), 외부 데이터 파일
    3) 노드를 PyTorch 모듈(RevIN, 패치 임베딩, 인코더 층, head)별로 묶어 보기
    4) 노드 메타데이터에 남은 소스 경로(stack trace) 찾아서 지우기

실행 (프로젝트 루트에서, 13장 결과 사용):
    python docs/manual/code/ch14_inspect_onnx.py
    python docs/manual/code/ch14_inspect_onnx.py --onnx outputs/manual/onnx/student.onnx --clean outputs/manual/onnx_clean/student.onnx
"""

import argparse
import re
from collections import Counter, OrderedDict
from pathlib import Path

import numpy as np
import onnx
from onnx import TensorProto, helper, numpy_helper

ROOT = Path(__file__).resolve().parents[3]
DTYPE = {TensorProto.FLOAT: "float32", TensorProto.INT64: "int64", TensorProto.BOOL: "bool"}


# ── 1) 손으로 만드는 가장 작은 ONNX ──────────────────────────────────────────
def build_tiny(path):
    """입력 x[1,2] → MatMul(W[2,2]) → Add(b[2]) → 출력 y[1,2]. 2장의 선형 분류기와 같은 계산."""
    W = numpy_helper.from_array(np.array([[1.0, -1.0], [0.5, 2.0]], dtype=np.float32), name="W")
    b = numpy_helper.from_array(np.array([0.1, -0.1], dtype=np.float32), name="b")
    nodes = [helper.make_node("MatMul", ["x", "W"], ["xw"], name="matmul"),
             helper.make_node("Add", ["xw", "b"], ["y"], name="add")]
    graph = helper.make_graph(
        nodes, "tiny_linear",
        inputs=[helper.make_tensor_value_info("x", TensorProto.FLOAT, [1, 2])],
        outputs=[helper.make_tensor_value_info("y", TensorProto.FLOAT, [1, 2])],
        initializer=[W, b])
    # ir_version: 파일 형식 버전. onnx 1.23은 기본값이 14인데 onnxruntime 1.30은 13까지만 읽는다.
    # PyTorch가 만든 파일과 같은 10으로 지정한다. (opset은 연산 버전, IR은 파일 형식 버전으로 서로 다르다)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 18)], producer_name="manual", ir_version=10)
    onnx.checker.check_model(model)
    onnx.save(model, path)
    return model


# ── 2) 실제 모델 구성 ────────────────────────────────────────────────────────
def shape_of(value_info):
    return [d.dim_value if d.dim_value else d.dim_param for d in value_info.type.tensor_type.shape.dim]


def numel(tensor):
    return int(np.prod(tensor.dims)) if tensor.dims else 1


def describe(path, top=5):
    model = onnx.load(path, load_external_data=False)      # 구조만 읽는다 (가중치 86 MB는 안 읽음)
    g = model.graph
    print(f"■ 헤더: IR 버전 {model.ir_version}, opset {[(o.domain or 'ai.onnx', o.version) for o in model.opset_import]}, "
          f"만든 도구 {model.producer_name} {model.producer_version}")
    print("■ 입력:", [(i.name, shape_of(i), DTYPE.get(i.type.tensor_type.elem_type)) for i in g.input])
    print("■ 출력:", [(o.name, shape_of(o), DTYPE.get(o.type.tensor_type.elem_type)) for o in g.output])

    ops = Counter(n.op_type for n in g.node)
    print(f"■ 노드(연산) {len(g.node)}개, 종류 {len(ops)}가지:")
    print("   ", ", ".join(f"{k} {v}" for k, v in ops.most_common()))

    total = sum(numel(t) for t in g.initializer)
    external = [t for t in g.initializer if t.data_location == TensorProto.EXTERNAL]
    print(f"■ 가중치(initializer) {len(g.initializer)}개, 숫자 {total:,}개 ({total * 4 / 1e6:.1f} MB)")
    print(f"   그중 외부 파일에 저장 {len(external)}개, .onnx 안에 직접 저장 {len(g.initializer) - len(external)}개(작은 상수)")
    if external:
        info = {e.key: e.value for e in external[0].external_data}
        print(f"   외부 데이터 예: {external[0].name} → 파일 '{info['location']}', 시작 {info['offset']}바이트, 길이 {info['length']}바이트")
    print(f"■ 가장 큰 가중치 {top}개:")
    for t in sorted(g.initializer, key=numel, reverse=True)[:top]:
        print(f"   {t.name:48s} {list(t.dims)}")
    return model


def module_of(node):
    """노드 메타데이터의 name_scopes로 어느 PyTorch 모듈에서 나온 연산인지 알아낸다."""
    scopes = {p.key: p.value for p in node.metadata_props}.get("pkg.torch.onnx.name_scopes", "")
    s = " ".join(re.findall(r"'([^']*)'", scopes))
    if "normalizer" in s:
        return "① normalizer (RevIN)"
    if "patch_embedding" in s or "tokenizer" in s:
        return "②③ 패치 + 임베딩"
 